# 03. Plate Detection Dataset & Baseline (U6–U7)

U6에서는 B00 영상의 다섯 구간에서 원판 bbox 학습 데이터를 만들고, U7에서는 작은 YOLO smoke 학습과 전체 후보·확인된 track을 생성하고 검토합니다.

## 범위와 완료 기준

이 노트북은 **프레임 추출 → 수동 ROI 라벨링 → contact sheet 검토 → smoke 학습 → 전체 후보 저장 → 첫 후보 확인 → track 저장·검토**를 담당합니다. 같은 이미지 풀을 `train`과 `val`에 사용하므로 smoke 결과를 성능이나 일반화 수치로 주장하지 않습니다.

## 1. Setup

재사용 로직은 `plate.py`, 영상 순회는 `video.py`에서 가져옵니다.

In [ ]:
from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from snatch_analysis.config import ARTIFACT_ROOT, B00_VIDEO_ID
from snatch_analysis.plate import (
    SAMPLE_PHASES,
    sample_phase_frames,
    save_plate_image,
    save_yolo_plate_label,
    yolo_bbox_to_opencv,
)
from snatch_analysis.video import iter_video_frames, read_video_properties


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요."
    )

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## 2. 조정할 설정

처음에는 `SAMPLE_COUNT=30`으로 유지하세요. 프레임을 확인한 뒤 라벨링할 때만 `RUN_MANUAL_LABELING=True`로 바꿉니다. 라벨 하나를 고칠 때는 `LABEL_FRAME_INDICES={프레임 번호}`와 `OVERWRITE_LABELS=True`를 함께 설정합니다. 모든 bbox를 contact sheet로 확인한 뒤에만 `CONTACT_SHEET_REVIEWED=True`로 바꿉니다.

In [ ]:
VIDEO_ID = B00_VIDEO_ID
SAMPLE_COUNT = 30  # 허용 범위: 20~40
RUN_MANUAL_LABELING = False
OVERWRITE_IMAGES = False
OVERWRITE_LABELS = False
LABEL_FRAME_INDICES = None  # 예: {65}; None이면 아직 라벨 없는 전체 샘플
CONTACT_SHEET_REVIEWED = True

MANIFEST_PATH = Path("data/metadata/videos.csv")
PHASE_SNAPSHOT_PATH = ARTIFACT_ROOT / VIDEO_ID / "phase_by_frame.csv"
DATASET_ROOT = Path("data/plate_dataset")
IMAGES_DIRECTORY = DATASET_ROOT / "images"
LABELS_DIRECTORY = DATASET_ROOT / "labels"
SAMPLE_MANIFEST_PATH = DATASET_ROOT / "sample_frames.csv"
DATASET_YAML_PATH = DATASET_ROOT / "dataset.yaml"
CONTACT_SHEET_PATH = ARTIFACT_ROOT / VIDEO_ID / "plate_label_contact_sheet.png"

if not 20 <= SAMPLE_COUNT <= 40:
    raise ValueError("SAMPLE_COUNT는 20~40이어야 합니다.")
if OVERWRITE_LABELS and not LABEL_FRAME_INDICES:
    raise ValueError(
        "덮어쓸 LABEL_FRAME_INDICES를 한 개 이상 지정하세요."
    )

## 3. 입력 검증

Manifest의 B00 경로와 U3에서 저장한 프레임별 구간표가 실제 영상과 일치하는지 확인합니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
video_rows = manifest.loc[manifest["video_id"] == VIDEO_ID]
if len(video_rows) != 1:
    raise ValueError(f"Manifest에서 {VIDEO_ID} 한 행을 찾을 수 없습니다.")

video_path = Path(video_rows.iloc[0]["path"])
video_properties = read_video_properties(video_path)
if video_properties["read_status"] != "ok":
    raise RuntimeError(f"영상을 읽을 수 없습니다: {video_path}")

phase_by_frame = pd.read_csv(PHASE_SNAPSHOT_PATH)
frame_count = int(video_properties["frame_count"])
if len(phase_by_frame) != frame_count:
    raise ValueError("구간표 행 수와 영상 frame_count가 다릅니다.")
if set(phase_by_frame["video_id"]) != {VIDEO_ID}:
    raise ValueError("구간표 video_id가 선택 영상과 다릅니다.")

pd.DataFrame(
    [{
        "video_id": VIDEO_ID,
        "video_path": video_path.as_posix(),
        **video_properties,
        "phase_rows": len(phase_by_frame),
    }]
)

## 4. 다섯 구간 분산 샘플링

각 구간 안에서 시간 순으로 고르게 선택하고 중복 프레임을 허용하지 않습니다. B00의 `bar_off_floor=0` 때문에 분석 구간상 Set-up 프레임이 비어 있으므로, **샘플링에서만 frame 0을 Set-up 기준 프레임으로 사용**합니다. 원래 `analysis_phase` 값은 바꾸지 않습니다.

In [ ]:
sampled_frames = sample_phase_frames(
    phase_by_frame,
    sample_count=SAMPLE_COUNT,
)
sampled_frames["image_filename"] = sampled_frames["frame_index"].map(
    lambda index: f"{VIDEO_ID}_frame_{index:04d}.jpg"
)
sampled_frames["label_filename"] = sampled_frames["frame_index"].map(
    lambda index: f"{VIDEO_ID}_frame_{index:04d}.txt"
)
if LABEL_FRAME_INDICES is not None:
    unknown_label_frames = set(LABEL_FRAME_INDICES) - set(
        sampled_frames["frame_index"]
    )
    if unknown_label_frames:
        raise ValueError(
            f"샘플에 없는 LABEL_FRAME_INDICES: {sorted(unknown_label_frames)}"
        )

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
sampled_frames.to_csv(SAMPLE_MANIFEST_PATH, index=False)

phase_distribution = (
    sampled_frames["sampling_phase"]
    .value_counts()
    .reindex(SAMPLE_PHASES, fill_value=0)
    .rename_axis("sampling_phase")
    .reset_index(name="sample_count")
)
display(phase_distribution)
sampled_frames

## 5. 프레임 추출

영상 전체를 메모리에 올리지 않고 한 번만 순회합니다. 기존 이미지는 기본적으로 유지합니다.

In [ ]:
sample_by_index = sampled_frames.set_index("frame_index")
pending_indices = {
    frame_index
    for frame_index, row in sample_by_index.iterrows()
    if OVERWRITE_IMAGES
    or not (IMAGES_DIRECTORY / row["image_filename"]).exists()
}
extraction_actions = [
    {"frame_index": frame_index, "action": "skipped"}
    for frame_index in sample_by_index.index
    if frame_index not in pending_indices
]
frame_iterator = (
    iter_video_frames(video_path, expected_frame_count=frame_count)
    if pending_indices
    else ()
)

for frame_index, _, frame in frame_iterator:
    if frame_index not in pending_indices:
        continue
    image_path = IMAGES_DIRECTORY / sample_by_index.loc[
        frame_index, "image_filename"
    ]
    action = save_plate_image(
        image_path,
        frame,
        overwrite=OVERWRITE_IMAGES,
    )
    extraction_actions.append(
        {"frame_index": frame_index, "action": action}
    )
    pending_indices.remove(frame_index)
    if not pending_indices:
        break

if pending_indices:
    raise RuntimeError(f"추출하지 못한 프레임: {sorted(pending_indices)}")
if len(extraction_actions) != SAMPLE_COUNT:
    raise RuntimeError("일부 샘플 프레임을 추출하지 못했습니다.")

pd.DataFrame(extraction_actions)["action"].value_counts().rename("count")

## 6. OpenCV 수동 bbox 라벨링

설정 셀에서 `RUN_MANUAL_LABELING=True`로 바꾼 뒤 이 셀을 실행하세요. 각 창에서 **선수가 들어 올리는, 카메라 쪽 원판 하나**만 드래그하고 `Enter` 또는 `Space`를 누릅니다. `c`는 현재 프레임을 건너뜁니다. 배경의 원판, 거울 속 원판, 반대쪽 원판은 라벨링하지 않습니다.

In [ ]:
label_actions = []

if RUN_MANUAL_LABELING:
    for row in sampled_frames.itertuples(index=False):
        if (
            LABEL_FRAME_INDICES is not None
            and row.frame_index not in LABEL_FRAME_INDICES
        ):
            continue
        image_path = IMAGES_DIRECTORY / row.image_filename
        label_path = LABELS_DIRECTORY / row.label_filename
        if label_path.exists() and not OVERWRITE_LABELS:
            label_actions.append(
                {"frame_index": row.frame_index, "action": "skipped"}
            )
            continue

        frame = cv2.imread(str(image_path))
        if frame is None:
            raise RuntimeError(f"추출 이미지를 읽을 수 없습니다: {image_path}")
        window_name = f"plate | frame {row.frame_index} | {row.sampling_phase}"
        bbox = cv2.selectROI(
            window_name,
            frame,
            showCrosshair=True,
            fromCenter=False,
        )
        cv2.destroyWindow(window_name)
        if bbox[2] == 0 or bbox[3] == 0:
            action = "cancelled"
        else:
            action = save_yolo_plate_label(
                label_path,
                bbox,
                image_width=frame.shape[1],
                image_height=frame.shape[0],
                overwrite=OVERWRITE_LABELS,
            )
        label_actions.append(
            {"frame_index": row.frame_index, "action": action}
        )
    cv2.destroyAllWindows()
else:
    print("수동 라벨링 대기: RUN_MANUAL_LABELING=True로 바꾸세요.")

pd.DataFrame(label_actions)

## 7. 모든 bbox contact sheet 검토

저장된 YOLO 좌표를 다시 pixel bbox로 복원해 한 화면에서 확인합니다. bbox가 원판을 벗어나거나 다른 물체를 가리키면 해당 프레임만 `OVERWRITE_LABELS=True`로 다시 라벨링하세요.

In [ ]:
contact_frames = []

for row in sampled_frames.itertuples(index=False):
    image_path = IMAGES_DIRECTORY / row.image_filename
    label_path = LABELS_DIRECTORY / row.label_filename
    if not label_path.exists():
        continue

    tokens = label_path.read_text(encoding="utf-8").strip().split()
    if len(tokens) != 5 or tokens[0] != "0":
        raise ValueError(f"잘못된 원판 라벨: {label_path}")
    yolo_bbox = tuple(float(value) for value in tokens[1:])

    frame = cv2.imread(str(image_path))
    if frame is None:
        raise RuntimeError(f"추출 이미지를 읽을 수 없습니다: {image_path}")
    x, y, width, height = yolo_bbox_to_opencv(
        yolo_bbox,
        image_width=frame.shape[1],
        image_height=frame.shape[0],
    )
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + width, y + height), (0, 255, 0), 4)
    thumbnail_width = 360
    thumbnail_height = round(overlay.shape[0] * thumbnail_width / overlay.shape[1])
    thumbnail = cv2.resize(overlay, (thumbnail_width, thumbnail_height))
    contact_frames.append((row, thumbnail))

if contact_frames:
    column_count = 5
    row_count = (len(contact_frames) + column_count - 1) // column_count
    figure, axes = plt.subplots(
        row_count,
        column_count,
        figsize=(20, 4 * row_count),
        squeeze=False,
    )
    for axis in axes.flat:
        axis.axis("off")
    for axis, (row, overlay) in zip(axes.flat, contact_frames):
        axis.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{row.frame_index} | {row.sampling_phase}")
    figure.tight_layout()
    CONTACT_SHEET_PATH.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(CONTACT_SHEET_PATH, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Contact sheet: {CONTACT_SHEET_PATH}")
else:
    print("저장된 라벨이 없어 contact sheet 생성을 기다립니다.")

## 8. YOLO dataset 설정과 수동 체크포인트

추가 YAML 라이브러리 없이 Ultralytics가 읽을 최소 설정만 저장합니다. U7로 넘어가기 전에 라벨 수가 샘플 수와 같고 contact sheet를 직접 확인해야 합니다.

In [ ]:
dataset_yaml = (
    f"path: {DATASET_ROOT.resolve().as_posix()}\n"
    "train: images\n"
    "val: images\n"
    "names:\n"
    "  0: plate\n"
)
DATASET_YAML_PATH.write_text(dataset_yaml, encoding="utf-8")

image_count = sum(
    (IMAGES_DIRECTORY / name).exists()
    for name in sampled_frames["image_filename"]
)
label_count = sum(
    (LABELS_DIRECTORY / name).exists()
    for name in sampled_frames["label_filename"]
)
if image_count != SAMPLE_COUNT or label_count != SAMPLE_COUNT:
    manual_checkpoint = "PENDING: 수동 bbox 라벨링 미완료"
elif not CONTACT_SHEET_PATH.exists() or not CONTACT_SHEET_REVIEWED:
    manual_checkpoint = "READY_FOR_REVIEW: contact sheet 직접 확인 필요"
else:
    manual_checkpoint = "PASS: 모든 bbox 직접 검토 완료"

pd.DataFrame(
    [{
        "sample_count": SAMPLE_COUNT,
        "image_count": image_count,
        "label_count": label_count,
        "manual_checkpoint": manual_checkpoint,
        "dataset_yaml": DATASET_YAML_PATH.as_posix(),
        "contact_sheet": (
            CONTACT_SHEET_PATH.as_posix()
            if CONTACT_SHEET_PATH.exists()
            else "N/A: 라벨링 대기"
        ),
    }]
)

## 9. U7 smoke 학습과 전체 후보 저장

검토 완료한 30개 라벨로 작은 smoke 학습을 실행할 수 있습니다. `train`과 `val`은 같은 이미지 풀이므로 이 결과는 성능이나 일반화 수치가 아니라 실행 가능성 확인용입니다. 추론에서는 모든 후보를 먼저 저장하고, 선택 track은 사람이 첫 후보를 확인한 뒤에만 만듭니다.

In [ ]:
import json
import time

from snatch_analysis.pose import (
    resolve_inference_device,
    verify_ultralytics_safe_load,
)
safe_load = verify_ultralytics_safe_load()

from ultralytics import YOLO

from snatch_analysis.config import DETECTION_MODEL_NAME
from snatch_analysis.plate import (
    plate_result_to_candidates,
    save_plate_candidate_cache,
    save_plate_track,
    select_primary_plate_track,
)



### 조정할 설정

기본값은 재학습하지 않고 이전 smoke weight를 사용합니다. 재학습은 라벨 검토가 끝난 경우에만 켜세요. `CONFIRM_PRIMARY_PLATE`는 preview에서 보인 첫 유효 frame과 candidate ID를 직접 입력한 뒤에만 `True`로 변경합니다.

In [ ]:
RETRAIN_PLATE = False
REQUESTED_DEVICE = "mps"
PLATE_EPOCHS = 10
PLATE_IMAGE_SIZE = 320
PLATE_BATCH = 4
PLATE_CONFIDENCE = 0.25

PLATE_RUN_DIRECTORY = ARTIFACT_ROOT / VIDEO_ID / "plate_smoke"
PLATE_WEIGHTS_PATH = PLATE_RUN_DIRECTORY / "weights" / "best.pt"
PLATE_CANDIDATES_PATH = ARTIFACT_ROOT / VIDEO_ID / "plate_candidates.csv"
PLATE_TRACK_PATH = ARTIFACT_ROOT / VIDEO_ID / "plate_track.csv"
RUN_SUMMARY_PATH = ARTIFACT_ROOT / VIDEO_ID / "run_summary.json"
POSE_KEYPOINT_PATH = ARTIFACT_ROOT / VIDEO_ID / "pose_keypoints.csv"

PRIMARY_PLATE_FRAME_INDEX = None
PRIMARY_PLATE_CANDIDATE_ID = None
CONFIRM_PRIMARY_PLATE = False
BACKGROUND_PLATE_MISSELECTION_FRAMES = ()  # 예: (42, 77); 수동 관찰, 성능 지표 아님

if not 0 < PLATE_EPOCHS <= 50 or not 0 < PLATE_BATCH <= 8:
    raise ValueError("M2 smoke 설정: epochs 1~50, batch 1~8")


### 학습 또는 기존 weight 확인

`.pt`를 읽기 전에 restricted safe-load를 확인합니다. 실제 MPS/CPU 장치와 학습 시간은 아래 실행 뒤 summary에 기록됩니다.

In [ ]:
plate_device = resolve_inference_device(REQUESTED_DEVICE)
training_seconds = 0.0

if RETRAIN_PLATE:
    if manual_checkpoint != "PASS: 모든 bbox 직접 검토 완료":
        raise RuntimeError("contact sheet 검토 완료 후에만 smoke 학습을 실행합니다.")
    training_started = time.perf_counter()
    training_result = YOLO(DETECTION_MODEL_NAME).train(
        data=str(DATASET_YAML_PATH),
        epochs=PLATE_EPOCHS,
        imgsz=PLATE_IMAGE_SIZE,
        batch=PLATE_BATCH,
        device=plate_device["actual_device"],
        project=str(PLATE_RUN_DIRECTORY.parent.resolve()),
        name=PLATE_RUN_DIRECTORY.name,
        exist_ok=True,
    )
    training_seconds = time.perf_counter() - training_started
    PLATE_WEIGHTS_PATH = Path(training_result.save_dir) / "weights" / "best.pt"

if not PLATE_WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"원판 weight가 없습니다: {PLATE_WEIGHTS_PATH}. RETRAIN_PLATE=True로 smoke 학습을 실행하세요."
    )

pd.DataFrame([{**plate_device, **safe_load, "weight_path": str(PLATE_WEIGHTS_PATH), "training_seconds": training_seconds}])


### 전체 frame 후보 추론

모든 bbox 후보를 `plate_candidates.csv`에 저장합니다. 후보가 없는 frame은 후보 표에 행이 없고, 다음 단계의 254행 track에서 `missing`으로 명시합니다.

In [ ]:
inference_started = time.perf_counter()
plate_model = YOLO(str(PLATE_WEIGHTS_PATH))
candidate_frames = []
for frame_index, result in enumerate(plate_model.predict(
    source=str(video_path),
    stream=True,
    conf=PLATE_CONFIDENCE,
    device=plate_device["actual_device"],
    verbose=False,
)):
    candidate_frames.append(plate_result_to_candidates(
        result, VIDEO_ID, frame_index, frame_index / float(video_properties["fps"]),
        image_width=int(video_properties["width"]),
        image_height=int(video_properties["height"]),
    ))
inference_seconds = time.perf_counter() - inference_started
if len(candidate_frames) != frame_count:
    raise RuntimeError(f"전체 frame 추론이 완료되지 않았습니다: {len(candidate_frames)}/{frame_count}")
plate_candidates = pd.concat(candidate_frames, ignore_index=True)
save_plate_candidate_cache(plate_candidates, PLATE_CANDIDATES_PATH)

run_summary = json.loads(RUN_SUMMARY_PATH.read_text()) if RUN_SUMMARY_PATH.exists() else {}
recorded_training_seconds = (
    training_seconds
    if RETRAIN_PLATE
    else float(run_summary.get("plate", {}).get("training_seconds", 0.0))
)
run_summary["plate"] = {
    "status": "candidates_complete",
    "actual_device": plate_device["actual_device"],
    "weight_path": str(PLATE_WEIGHTS_PATH),
    "training_seconds": recorded_training_seconds,
    "inference_seconds": inference_seconds,
    "processed_frame_count": len(candidate_frames),
    "candidate_count": len(plate_candidates),
    "background_plate_misselection_frames": list(BACKGROUND_PLATE_MISSELECTION_FRAMES),
}
RUN_SUMMARY_PATH.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
pd.DataFrame([run_summary["plate"]])


### 첫 유효 후보 확인

아래 preview의 숫자는 candidate ID입니다. 들어 올리는 카메라 쪽 원판의 ID를 설정 셀에 기록하세요. 배경·거울·반대쪽 원판이 비슷하면 확정하지 말고 `CONFIRM_PRIMARY_PLATE=False`로 둡니다.

In [ ]:
if plate_candidates.empty:
    raise RuntimeError("원판 후보가 없어 track을 만들 수 없습니다.")
first_candidate_frame = int(plate_candidates["frame_index"].min())
candidate_phase = (
    plate_candidates[["frame_index"]]
    .drop_duplicates()
    .merge(phase_by_frame[["frame_index", "phase"]], on="frame_index", how="left")
)
phase_review_frame_indices = {}
for phase in SAMPLE_PHASES:
    fallback_frame = int(
        sampled_frames.loc[
            sampled_frames["sampling_phase"] == phase, "frame_index"
        ].iloc[0]
    )
    detected_phase_frames = sorted(
        candidate_phase.loc[candidate_phase["phase"] == phase, "frame_index"]
    )
    phase_review_frame_indices[phase] = (
        int(detected_phase_frames[len(detected_phase_frames) // 2])
        if detected_phase_frames
        else fallback_frame
    )
review_frame_indices = {first_candidate_frame, *phase_review_frame_indices.values()}
if len(review_frame_indices) > 6:
    raise RuntimeError('review frame cache는 최대 6개여야 합니다.')
review_frames = {}
for frame_index, _, frame in iter_video_frames(video_path, expected_frame_count=frame_count):
    if frame_index in review_frame_indices:
        review_frames[frame_index] = frame
    if len(review_frames) == len(review_frame_indices):
        break
if set(review_frames) != review_frame_indices:
    raise RuntimeError(f'검토 frame을 읽지 못했습니다: {sorted(review_frame_indices - set(review_frames))}')
preview_candidates = plate_candidates.loc[
    plate_candidates["frame_index"] == first_candidate_frame
].copy()
preview = review_frames[first_candidate_frame].copy()
for candidate in preview_candidates.itertuples(index=False):
    cv2.rectangle(preview, (round(candidate.bbox_x1), round(candidate.bbox_y1)), (round(candidate.bbox_x2), round(candidate.bbox_y2)), (0, 255, 0), 2)
    cv2.putText(preview, str(candidate.candidate_id), (round(candidate.bbox_x1), round(candidate.bbox_y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"first candidate frame={first_candidate_frame}; set this frame and one ID above")
display(preview_candidates)


### 다섯 구간 후보 검토

각 구간의 기존 샘플 한 frame에서 모든 후보 bbox와 중심을 확인합니다. `BACKGROUND_PLATE_MISSELECTION_FRAMES`에는 배경 원판을 잘못 고른 것으로 관찰한 frame만 기록하세요. 이 수동 관찰은 성능 지표나 일반화 주장이 아닙니다.

In [ ]:
review_rows = []
figure, axes = plt.subplots(1, len(SAMPLE_PHASES), figsize=(24, 5), squeeze=False)
for axis, phase in zip(axes.flat, SAMPLE_PHASES):
    review_frame_index = phase_review_frame_indices[phase]
    review_frame = review_frames[review_frame_index].copy()
    phase_candidates = plate_candidates.loc[plate_candidates['frame_index'] == review_frame_index]
    for candidate in phase_candidates.itertuples(index=False):
        cv2.rectangle(review_frame, (round(candidate.bbox_x1), round(candidate.bbox_y1)), (round(candidate.bbox_x2), round(candidate.bbox_y2)), (0, 255, 0), 2)
        cv2.circle(review_frame, (round(candidate.center_x_px), round(candidate.center_y_px)), 4, (255, 0, 0), -1)
        cv2.putText(review_frame, f'{candidate.candidate_id}', (round(candidate.bbox_x1), round(candidate.bbox_y1) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    axis.imshow(cv2.cvtColor(review_frame, cv2.COLOR_BGR2RGB))
    axis.set_title(f'{phase} | frame {review_frame_index} | {len(phase_candidates)} candidates')
    axis.axis('off')
    review_rows.append({'phase': phase, 'frame_index': review_frame_index, 'candidate_count': len(phase_candidates), 'background_plate_misselection_marked': review_frame_index in BACKGROUND_PLATE_MISSELECTION_FRAMES})
figure.tight_layout()
display(pd.DataFrame(review_rows))
display(pd.DataFrame({'background_plate_misselection_frame': list(BACKGROUND_PLATE_MISSELECTION_FRAMES)}))


### 확인 후 254행 track 저장

`CONFIRM_PRIMARY_PLATE=True` 전에는 파일을 저장하지 않습니다. 선택은 이전 중심·bbox 크기·주동작 선수 주변 위치를 함께 사용하고, 근접한 두 후보가 불명확하면 그 frame부터 `confirmation_required`로 중단합니다. 좌표는 보간하지 않아 plot의 빈 구간이 그대로 남습니다.

In [ ]:
if not CONFIRM_PRIMARY_PLATE:
    print("track 미저장: preview를 확인한 뒤 CONFIRM_PRIMARY_PLATE=True로 바꾸세요.")
elif PRIMARY_PLATE_FRAME_INDEX is None or PRIMARY_PLATE_CANDIDATE_ID is None:
    raise ValueError("확인한 PRIMARY_PLATE_FRAME_INDEX와 PRIMARY_PLATE_CANDIDATE_ID를 입력하세요.")
else:
    if PRIMARY_PLATE_FRAME_INDEX != first_candidate_frame:
        raise ValueError(f"첫 후보 frame {first_candidate_frame}만 확인할 수 있습니다.")
    if POSE_KEYPOINT_PATH.exists():
        lifter_centers = pd.read_csv(POSE_KEYPOINT_PATH).loc[:, ["frame_index", "bbox_center_x", "bbox_center_y"]].drop_duplicates("frame_index").rename(columns={"bbox_center_x": "center_x_px", "bbox_center_y": "center_y_px"})
    else:
        lifter_centers = None
        print("선수 Pose가 없어 선수 주변 위치 비용 없이 track을 만듭니다.")
    plate_track = select_primary_plate_track(
        plate_candidates, frame_count=frame_count,
        initial_frame_index=PRIMARY_PLATE_FRAME_INDEX,
        initial_candidate_id=PRIMARY_PLATE_CANDIDATE_ID,
        lifter_centers=lifter_centers, fps=float(video_properties["fps"]),
    )
    save_plate_track(plate_track, PLATE_TRACK_PATH)
    run_summary['plate'].update(status='track_complete', track_path=str(PLATE_TRACK_PATH), background_plate_misselection_frames=list(BACKGROUND_PLATE_MISSELECTION_FRAMES))
    longest_non_ok_gap = current_non_ok_gap = 0
    for status in plate_track['status']:
        current_non_ok_gap = current_non_ok_gap + 1 if status != 'ok' else 0
        longest_non_ok_gap = max(longest_non_ok_gap, current_non_ok_gap)
    run_summary['plate']['longest_consecutive_non_ok_gap_frames'] = longest_non_ok_gap
    RUN_SUMMARY_PATH.write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
    display(pd.DataFrame([{'longest_consecutive_non_ok_gap_frames': longest_non_ok_gap}]))
    plt.plot(plate_track["frame_index"], plate_track["center_x_normalized"], marker=".")
    plt.xlabel("frame index")
    plt.ylabel("plate center x (normalized)")
    plt.title("Plate trajectory; missing rows remain gaps")
    plt.show()
    display(plate_track["status"].value_counts(dropna=False))


### 선택 track의 다섯 구간 검토

확인 후 저장된 track만 검토합니다. 각 기존 구간 샘플에서 선택 bbox와 중심을 보이고, 선택되지 못한 frame은 status/reason을 표시합니다. 빨간 표시는 수동으로 기록한 배경 원판 오선택 관찰입니다.

In [ ]:
if not CONFIRM_PRIMARY_PLATE:
    print('선택 track 검토 대기: 후보를 확인하고 CONFIRM_PRIMARY_PLATE=True로 저장하세요.')
else:
    selected_review_rows = []
    figure, axes = plt.subplots(1, len(SAMPLE_PHASES), figsize=(24, 5), squeeze=False)
    for axis, phase in zip(axes.flat, SAMPLE_PHASES):
        review_frame_index = phase_review_frame_indices[phase]
        review_frame = review_frames[review_frame_index].copy()
        candidate = plate_track.loc[plate_track['frame_index'] == review_frame_index].iloc[0]
        marked_background = review_frame_index in BACKGROUND_PLATE_MISSELECTION_FRAMES
        if candidate.status == 'ok':
            cv2.rectangle(review_frame, (round(candidate.bbox_x1), round(candidate.bbox_y1)), (round(candidate.bbox_x2), round(candidate.bbox_y2)), (0, 255, 0), 2)
            cv2.circle(review_frame, (round(candidate.center_x_px), round(candidate.center_y_px)), 4, (255, 0, 0), -1)
            detail = 'ok'
        else:
            detail = f'{candidate.status}: {candidate.reason}'
            cv2.putText(review_frame, detail, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        if marked_background:
            cv2.putText(review_frame, 'BACKGROUND MISELECTION', (20, review_frame.shape[0] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        axis.imshow(cv2.cvtColor(review_frame, cv2.COLOR_BGR2RGB))
        axis.set_title(f'{phase} | {detail}')
        axis.axis('off')
        selected_review_rows.append({'phase': phase, 'frame_index': review_frame_index, 'status': candidate.status, 'reason': candidate.reason, 'background_plate_misselection_marked': marked_background})
    figure.tight_layout()
    display(pd.DataFrame(selected_review_rows))
